In [84]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [85]:
# build gridworld environment
class Gridworld:
    def __init__(self, size_x=4, size_y=4, goal=None, slip_prob=0.0):
        self.size_x = size_x
        self.size_y = size_y
        # States
        self.states = [(x, y) for x in range(size_x) for y in range(size_y)]
        # Initial State
        self.initial_state = (0, 0)
        # Terminal States
        self.terminal_states = [(size_x - 1, size_y - 1)] if goal is None else [goal]
        self.slip_prob = slip_prob

        # Actions
        self.actions = ['up', 'down', 'left', 'right']
        # Transition Model
        self.transition_model = self.stochastic_transitions
        # Reward Function
        self.reward_function = self.simple_reward
        # Discount Factor
        self.discount_factor = 0.9

    def deterministic_transitions(self, state, action):
        # Deterministic transition model
        if state in self.terminal_states:
            return {state: 1.0}  # No transitions from terminal states

        if action == 'up':
            candidate = (state[0] - 1, state[1])
           
        elif action == 'down':
            candidate = (state[0] + 1, state[1])
  
        elif action == 'left':
            candidate = (state[0], state[1] - 1)
      
        elif action == 'right':
            candidate = (state[0], state[1] + 1)
       
        else:
            candidate = state

        x, y = candidate
        if 0 <= x < self.size_x and 0 <= y < self.size_y:
            return {candidate: 1.0}
        else:
            return {state: 1.0}

    def stochastic_transitions(self, state, action):
        # Stochastic transition fuction that returns a distribution over next states
        if state in self.terminal_states:
            return {state: 1.0}  # No transitions from terminal states

        next_states = {}
        for a in self.actions:
            if a == action:
                next_state = self._deterministic_next(state, a)
                next_states[next_state] = next_states.get(next_state, 0) + (1 - self.slip_prob)
            else:
                next_state = self._deterministic_next(state, a)
                next_states[next_state] = next_states.get(next_state, 0) + self.slip_prob / (len(self.actions) - 1)

        return next_states

    def simple_reward(self, state, action, next_state):
        if next_state in self.terminal_states:
            return 1  # Reward for reaching terminal state
        else:
            return -0.1  # Small negative reward for each step taken

    def value_iteration(self, threshold=0.0001):
        converged = False
        U = {state: 0 for state in self.states}  # Initialize utilities

        while not converged: 
            U_new = {}
            for state in self.states:
                if state in self.terminal_states:
                    U_new[state] = 0
                    continue
                
                # Compute Bellman Update

                U_new[state] = max(
                sum(
                    prob * (self.reward_function(state, action, next_state) + self.discount_factor * U[next_state])
                    for next_state, prob in self.transition_model(state, action).items()
                )
                for action in self.actions
            )
                                   
            # Check for convergence
            if max(abs(U_new[state] - U[state]) for state in self.states) < threshold:
                converged = True
            U = U_new
        return U

    def policy_extraction(self, U):

        policy = {}
        for state in self.states:
            if state in self.terminal_states:
                policy[state] = None
                continue

            best_action = None
            best_value =  float('-inf')

            for action in self.actions:
                next_states = self.transition_model(state, action)
                value = sum(
                    prob * (self.reward_function(state, action, next_state) + self.discount_factor * U[next_state])
                    for next_state, prob in next_states.items()
                )
                if value > best_value:
                    best_value = value
                    best_action = action

            policy[state] = best_action
        return policy

    def policy_iteration(self, threshold=0.0001):
        # Initialize a random policy
        policy = {state: np.random.choice(self.actions) for state in self.states if state not in self.terminal_states}
        policy.update({state: None for state in self.terminal_states})  # No actions for terminal states

        while True:
            # Policy Evaluation
            U = {state: 0 for state in self.states}  # Initialize utilities
            while True:
                U_new = {}
                for state in self.states:
                    if state in self.terminal_states:
                        U_new[state] = 0
                        continue
                    
                    action = policy[state]
                    U_new[state] = sum(
                        prob * (self.reward_function(state, action, next_state) + self.discount_factor * U[next_state])
                        for next_state, prob in self.transition_model(state, action).items()
                    )
                if max(abs(U_new[state] - U[state]) for state in self.states) < threshold:
                    break
                U = U_new

            new_policy = self.policy_extraction(U)
            policy_stable = all(
                policy[s] == new_policy[s] 
                for s in self.states 
                if s not in self.terminal_states
            )
            policy = new_policy
            if policy_stable:
                break

        return policy
        


    def trace_generation(self, policy, start_state=None, max_steps=100):
        if start_state is None:
            start_state = self.initial_state
        trace = []
        current_state = start_state 
        for _ in range(max_steps):
            action = policy[current_state]
            if action is None:  # Reached terminal state
                break
            next_state = self.sample_transition(current_state, action)
            reward = self.reward_function(current_state, action, next_state)
            trace.append((current_state, action, reward, next_state))
            current_state = next_state
        return trace

    def sample_transition(self, state, action):
        distribution = self.transition_model(state, action)
        next_states = list(distribution.keys())
        probabilities = list(distribution.values())
        idx = np.random.choice(len(next_states), p=probabilities)
        return next_states[idx]
   

    def _deterministic_next(self, state, action):
        return next(iter(self.deterministic_transitions(state, action).keys()))


    def trace_generation_random(self, start_state=None, max_steps=100, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        if start_state is None:
            start_state = self.initial_state
        trace = []
        current_state = start_state
        for _ in range(max_steps):
            if current_state in self.terminal_states:
                break
            action = rng.choice(self.actions)  # fresh random action per step
            next_state = self.sample_transition(current_state, action)
            reward = self.reward_function(current_state, action, next_state)
            trace.append((current_state, action, reward, next_state))
            current_state = next_state
        return trace

    def trace_generation_eps_greedy(self, optimal_policy, epsilon, start_state=None, max_steps=100, rng=None):
        if rng is None:
            rng = np.random.default_rng()
        if start_state is None:
            start_state = self.initial_state
        trace = []
        current_state = start_state
        for _ in range(max_steps):
            if current_state in self.terminal_states:
                break
            if rng.random() < epsilon:
                action = rng.choice(self.actions)
            else:
                action = optimal_policy[current_state]
            next_state = self.sample_transition(current_state, action)
            reward = self.reward_function(current_state, action, next_state)
            trace.append((current_state, action, reward, next_state))
            current_state = next_state
        return trace

In [ ]:
def generate_dataset(num_mdps=20, traces_per_policy=50, max_steps=100, seed=None):
    rng = np.random.default_rng(seed)
    mdps_metadata = []
    traces = []
    
    for mdp_idx in range(num_mdps):
        # generate one MDP from the family
        mdp_seed = int(rng.integers(0, 2**31))
        mdp = random_mdp(seed=mdp_seed)
        
        mdps_metadata.append({
            'mdp_idx': mdp_idx,
            'size_x': mdp.size_x,
            'size_y': mdp.size_y,
            'goal': mdp.terminal_states[0],
            'slip_prob': mdp.slip_prob,
            'mdp_seed': mdp_seed
        })
        
        # compute optimal policy once
        U = mdp.value_iteration()
        optimal_policy = mdp.policy_extraction(U)
        
        # generate traces for each policy type
        for _ in range(traces_per_policy):
            trace_seed = int(rng.integers(0, 2**31))
            trace_rng = np.random.default_rng(trace_seed)
            
            opt_trace = mdp.trace_generation(optimal_policy, max_steps=max_steps)
            eps_trace = mdp.trace_generation_eps_greedy(optimal_policy, epsilon=0.1, max_steps=max_steps, rng=trace_rng)
            rand_trace = mdp.trace_generation_random(max_steps=max_steps, rng=trace_rng)
            
            traces.append({'mdp_idx': mdp_idx, 'policy_type': 'optimal', 'trace': opt_trace})
            traces.append({'mdp_idx': mdp_idx, 'policy_type': 'eps_greedy_01', 'trace': eps_trace})
            traces.append({'mdp_idx': mdp_idx, 'policy_type': 'random', 'trace': rand_trace})
    
    return {'mdps': mdps_metadata, 'traces': traces}

In [87]:
import pickle
with open('dataset.pkl', 'wb') as f:
    pickle.dump(dataset, f)

# load
with open('dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

In [88]:
total_traces = len(dataset['traces'])
print(f"Total MDPs: {len(dataset['mdps'])}")
print(f"Total Traces: {total_traces}")

Total MDPs: 10
Total Traces: 600


In [89]:
from collections import defaultdict

# Trace counts and average lengths per policy type
counts = defaultdict(int)
total_lengths = defaultdict(int)

for entry in dataset['traces']:
    pt = entry['policy_type']
    counts[pt] += 1
    total_lengths[pt] += len(entry['trace'])

for pt in counts:
    avg = total_lengths[pt] / counts[pt]
    print(f"{pt}: {counts[pt]} traces, avg length {avg:.1f}")

optimal: 200 traces, avg length 4.6
eps_greedy_01: 200 traces, avg length 4.8
random: 200 traces, avg length 29.9


In [90]:
incomplete = defaultdict(int)
for entry in dataset['traces']:
    trace = entry['trace']
    if len(trace) == 0:
        incomplete[entry['policy_type']] += 1  # immediate terminal? shouldn't happen
        continue
    last_step = trace[-1]
    last_next_state = tuple(last_step[3])
    mdp_idx = entry['mdp_idx']
    goal = tuple(dataset['mdps'][mdp_idx]['goal'])
    if last_next_state != goal:
        incomplete[entry['policy_type']] += 1

print("Traces that didn't reach goal:")
for pt, n in incomplete.items():
    print(f"  {pt}: {n}/{counts[pt]}")
    

Traces that didn't reach goal:
  random: 8/200


In [91]:
def build_vocab(max_grid_size, actions):
    vocab = {}
    # state coordinates
    for i in range(max_grid_size):
        vocab[f'coord_{i}'] = len(vocab)
    # actions
    for a in actions:
        vocab[f'action_{a}'] = len(vocab)
    # rewards (you have exactly 2 distinct values in your dataset)
    vocab['reward_step'] = len(vocab)  # for -0.1
    vocab['reward_goal'] = len(vocab)  # for 1
    # special
    vocab['<eos>'] = len(vocab)
    return vocab

def tokenize_trace(trace, vocab):
    tokens = []
    for step in trace:
        state, action, reward, next_state = step
        tokens.append(vocab[f'coord_{state[0]}'])
        tokens.append(vocab[f'coord_{state[1]}'])
        tokens.append(vocab[f'action_{action}'])
        tokens.append(vocab['reward_step'] if reward < 0 else vocab['reward_goal'])
        tokens.append(vocab[f'coord_{next_state[0]}'])
        tokens.append(vocab[f'coord_{next_state[1]}'])
    tokens.append(vocab['<eos>'])
    return tokens

In [92]:

vocab = build_vocab(max_grid_size=5, actions=['up', 'down', 'left', 'right'])
test = tokenize_trace(dataset['traces'][0]['trace'], vocab)
test

[0, 0, 6, 9, 1, 0, 1, 0, 6, 10, 2, 0, 11]

In [93]:
all_tokenized = []
for entry in dataset['traces']:
    tokens = tokenize_trace(entry['trace'], vocab)
    all_tokenized.append({
        'tokens': tokens,
        'mdp_idx': entry['mdp_idx'],
        'policy_type': entry['policy_type'],
    })

lengths = [len(t['tokens']) for t in all_tokenized]
print(f"Total tokenized traces: {len(all_tokenized)}")
print(f"Token sequence lengths: min={min(lengths)}, max={max(lengths)}, mean={sum(lengths)/len(lengths):.1f}")
print(f"Vocabulary size: {len(vocab)}")

Total tokenized traces: 600
Token sequence lengths: min=7, max=601, mean=79.5
Vocabulary size: 12


In [94]:
from collections import defaultdict, Counter

# Empirical transition counts: for each (mdp_idx, state, action), count next_states
empirical = defaultdict(Counter)

for entry in dataset['traces']:
    mdp_idx = entry['mdp_idx']
    for step in entry['trace']:
        state, action, reward, next_state = step
        key = (mdp_idx, tuple(state), action)
        empirical[key][tuple(next_state)] += 1

# Pick one MDP and compare empirical vs true distribution
mdp_idx = 0
mdp = random_mdp(seed=...)  # you'd need to recreate the MDP, or save it differently

# For one specific (state, action), compare:
state = (1, 1)  # or whatever
action = 'down'
true_dist = mdp.stochastic_transitions(state, action)
emp_counts = empirical[(mdp_idx, state, action)]
total = sum(emp_counts.values())
emp_dist = {s: c/total for s, c in emp_counts.items()}

print(f"True: {true_dist}")
print(f"Empirical (n={total}): {emp_dist}")

TypeError: SeedSequence expects int or sequence of ints for entropy not Ellipsis